# Traffic Demand Prediction — v3 (Push Beyond 91%)
**Score = max(0, 100 × R²)**

v3 improvements over v2 (91.06%):
- CatBoost as 3rd gradient boosting model
- Stacking ensemble with Ridge meta-learner
- More interaction features (hour×geo, temp×weather, polynomial temp)
- K-Fold target encoding (less leakage)
- 80 Optuna trials per model
- Temperature binning
- More granular time features

In [1]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge
from sklearn import metrics
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


In [2]:
# ── 2. Load Data ──────────────────────────────────────────────────────────────
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
display(train.head())
print('Missing (train):', train.isnull().sum().to_dict())
print('Missing (test) :', test.isnull().sum().to_dict())

Train shape: (77299, 11)
Test shape : (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


Missing (train): {'Index': 0, 'geohash': 0, 'day': 0, 'timestamp': 0, 'demand': 0, 'RoadType': 600, 'NumberofLanes': 0, 'LargeVehicles': 0, 'Landmarks': 0, 'Temperature': 2495, 'Weather': 797}
Missing (test) : {'Index': 0, 'geohash': 0, 'day': 0, 'timestamp': 0, 'RoadType': 324, 'NumberofLanes': 0, 'LargeVehicles': 0, 'Landmarks': 0, 'Temperature': 1349, 'Weather': 431}


In [3]:
# ── 3. Feature Engineering ────────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy()

    # --- Timestamp: format is "H:M" ---
    ts_parts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour']   = ts_parts[0].astype(int)
    df['minute'] = ts_parts[1].astype(int)

    # --- Continuous time ---
    df['time_minutes'] = df['hour'] * 60 + df['minute']

    # --- Cyclical encoding ---
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['time_sin']   = np.sin(2 * np.pi * df['time_minutes'] / 1440)
    df['time_cos']   = np.cos(2 * np.pi * df['time_minutes'] / 1440)

    # --- Rush-hour flags ---
    df['morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 10)).astype(int)
    df['evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 20)).astype(int)
    df['night']        = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)
    df['midday']       = ((df['hour'] >= 11) & (df['hour'] <= 16)).astype(int)
    df['rush_hour']    = ((df['morning_rush'] == 1) | (df['evening_rush'] == 1)).astype(int)

    # --- Time period buckets ---
    df['time_period'] = pd.cut(df['hour'],
                               bins=[-1, 5, 10, 15, 20, 24],
                               labels=['night', 'morning', 'midday', 'evening', 'late_night'])

    # --- Geohash features ---
    df['geo_prefix']  = df['geohash'].astype(str).str[:4]
    df['geo_prefix3'] = df['geohash'].astype(str).str[:3]
    df['geo_prefix5'] = df['geohash'].astype(str).str[:5]

    # --- Temperature features ---
    df['temp_missing'] = df['Temperature'].isnull().astype(int)
    # Temperature bins (cold, mild, warm, hot)
    df['temp_bin'] = pd.cut(df['Temperature'],
                            bins=[-np.inf, 10, 20, 30, np.inf],
                            labels=['cold', 'mild', 'warm', 'hot'])
    # Temperature squared (non-linear effect)
    df['temp_squared'] = df['Temperature'] ** 2

    # --- RoadType missing flag ---
    df['road_missing'] = df['RoadType'].isnull().astype(int)

    # --- Hour squared (non-linear time effect) ---
    df['hour_squared'] = df['hour'] ** 2

    return df

train = engineer_features(train)
test  = engineer_features(test)
print('After feature engineering:', train.shape)

After feature engineering: (77299, 34)


In [4]:
# ── 4. K-Fold Target Encoding (less leakage) ──────────────────────────────────
global_mean = train['demand'].mean()

def kfold_target_encode(train_df, test_df, col, target='demand', n_folds=5, smoothing=20):
    """K-Fold target encoding to reduce leakage."""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # For test: use full train mean
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['smoothed'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    test_mapping = stats['smoothed'].to_dict()
    test_df[f'{col}_target_enc'] = test_df[col].map(test_mapping).fillna(global_mean)
    
    # For train: K-Fold encoding
    kf_enc = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    train_df[f'{col}_target_enc'] = np.nan
    
    for train_idx, val_idx in kf_enc.split(train_df):
        fold_stats = train_df.iloc[train_idx].groupby(col)[target].agg(['mean', 'count'])
        fold_stats['smoothed'] = (fold_stats['count'] * fold_stats['mean'] + smoothing * global_mean) / (fold_stats['count'] + smoothing)
        fold_mapping = fold_stats['smoothed'].to_dict()
        train_df.loc[val_idx, f'{col}_target_enc'] = train_df.iloc[val_idx][col].map(fold_mapping).fillna(global_mean)
    
    return train_df, test_df

# Apply K-Fold target encoding
train, test = kfold_target_encode(train, test, 'geohash', smoothing=15)
train, test = kfold_target_encode(train, test, 'geo_prefix', smoothing=30)
train, test = kfold_target_encode(train, test, 'geo_prefix3', smoothing=50)
train, test = kfold_target_encode(train, test, 'geo_prefix5', smoothing=25)

# Weather and RoadType target encoding
train, test = kfold_target_encode(train, test, 'Weather', smoothing=40)
train, test = kfold_target_encode(train, test, 'RoadType', smoothing=40)

print('K-Fold target encoding done.')
print(train[['geohash_target_enc', 'geo_prefix_target_enc', 'Weather_target_enc']].describe())

K-Fold target encoding done.
       geohash_target_enc  geo_prefix_target_enc  Weather_target_enc
count        77299.000000           77299.000000        77299.000000
mean             0.097383               0.093988            0.093908
std              0.098246               0.020940            0.000848
min              0.026154               0.026693            0.091767
25%              0.044738               0.097486            0.093444
50%              0.064256               0.102751            0.093991
75%              0.105227               0.103073            0.094458
max              0.835580               0.141653            0.095060


In [5]:
# ── 5. Interaction Features (after target encoding) ──────────────────────────
def add_interactions(df):
    df = df.copy()
    # Hour × geohash target encoding
    df['hour_x_geo'] = df['hour'] * df['geohash_target_enc']
    # Day × hour
    df['day_x_hour'] = df['day'] * df['hour']
    # Temperature × weather encoding
    df['temp_x_weather'] = df['Temperature'] * df['Weather_target_enc']
    # Lanes × large vehicles
    df['lanes_x_vehicles'] = df['NumberofLanes'] * (df['LargeVehicles'] == 'Allowed').astype(int)
    # Rush hour × geohash
    df['rush_x_geo'] = df['rush_hour'] * df['geohash_target_enc']
    # Time minutes × geo prefix
    df['time_x_geo_prefix'] = df['time_minutes'] * df['geo_prefix_target_enc']
    return df

train = add_interactions(train)
test  = add_interactions(test)
print('Interaction features added. Shape:', train.shape)

Interaction features added. Shape: (77299, 46)


In [6]:
# ── 6. Encode Categoricals ────────────────────────────────────────────────────
CAT_COLS = ['geohash', 'geo_prefix', 'geo_prefix3', 'geo_prefix5', 'RoadType',
            'LargeVehicles', 'Landmarks', 'Weather', 'time_period', 'temp_bin']

combined = pd.concat([train, test], axis=0, ignore_index=True)

for col in CAT_COLS:
    if col in combined.columns:
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col].astype(str))

train = combined.iloc[:len(train)].copy()
test  = combined.iloc[len(train):].copy()
print('Label encoding done. Shape:', train.shape)

Label encoding done. Shape: (77299, 46)


In [7]:
# ── 7. Define Features & Handle Missing ───────────────────────────────────────
DROP_COLS = ['Index', 'demand', 'timestamp']  # Keep 'day'!
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS]
y      = train['demand']
X_test = test[FEATURE_COLS]

# Impute with train medians
train_medians = X.median(numeric_only=True)
X      = X.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

print(f'Features: {len(FEATURE_COLS)}')
print(f'X: {X.shape} | y: {y.shape} | X_test: {X_test.shape}')
print('NaN check:', X.isnull().sum().sum(), X_test.isnull().sum().sum())

Features: 43
X: (77299, 43) | y: (77299,) | X_test: (41778, 43)
NaN check: 0 0


In [8]:
# ── 8. Optuna Tuning: LightGBM ────────────────────────────────────────────────
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 800, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'num_leaves': trial.suggest_int('num_leaves', 63, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 80),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

print('Tuning LightGBM (80 trials)...')
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=80, show_progress_bar=True)
print(f'Best LightGBM CV R²: {study_lgb.best_value:.6f}')
print(f'Params: {study_lgb.best_params}')

Tuning LightGBM (80 trials)...


  0%|          | 0/80 [00:00<?, ?it/s]

Best LightGBM CV R²: 0.875994
Params: {'n_estimators': 839, 'learning_rate': 0.011230972407429885, 'max_depth': 9, 'num_leaves': 273, 'min_child_samples': 6, 'subsample': 0.7957461826806245, 'colsample_bytree': 0.6947260302758738, 'reg_alpha': 2.5915095944307844e-06, 'reg_lambda': 8.956150605627724e-08}


In [9]:
# ── 9. Optuna Tuning: XGBoost ─────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 800, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbosity': 0,
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

print('Tuning XGBoost (80 trials)...')
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=80, show_progress_bar=True)
print(f'Best XGBoost CV R²: {study_xgb.best_value:.6f}')
print(f'Params: {study_xgb.best_params}')

Tuning XGBoost (80 trials)...


  0%|          | 0/80 [00:00<?, ?it/s]

Best XGBoost CV R²: 0.877070
Params: {'n_estimators': 1739, 'learning_rate': 0.04990039648411618, 'max_depth': 10, 'min_child_weight': 1, 'subsample': 0.7332390290797127, 'colsample_bytree': 0.7552428690332091, 'reg_alpha': 0.0181632816039939, 'reg_lambda': 5.7792818568061426e-06, 'gamma': 0.001639865953216021}


In [12]:
# ── 10. Optuna Tuning: CatBoost ───────────────────────────────────────────────
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 800, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'depth': trial.suggest_int('depth', 5, 12),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 80),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'random_seed': 42,
        'verbose': 0,
        'thread_count': -1,
    }
    model = CatBoostRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

print('Tuning CatBoost (50 trials)...')
study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=True)
print(f'Best CatBoost CV R²: {study_cat.best_value:.6f}')
print(f'Params: {study_cat.best_params}')

Tuning CatBoost (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

Best CatBoost CV R²: 0.870416
Params: {'iterations': 1454, 'learning_rate': 0.010432057771114668, 'depth': 9, 'l2_leaf_reg': 0.043801839608120195, 'min_data_in_leaf': 41, 'subsample': 0.9442742961790465}


In [13]:
# ── 11. K-Fold Stacking Ensemble ──────────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Prepare best params
lgb_params = {**study_lgb.best_params, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
xgb_params = {**study_xgb.best_params, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}
cat_params = {**study_cat.best_params, 'random_seed': 42, 'verbose': 0, 'thread_count': -1}

# OOF predictions for stacking
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))
oof_rf  = np.zeros(len(X))

# Test predictions (averaged across folds)
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_rf  = np.zeros(len(X_test))

print('Training K-Fold ensemble...')
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # LightGBM
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(X_tr, y_tr)
    oof_lgb[val_idx] = m.predict(X_val)
    test_lgb += m.predict(X_test) / 5
    
    # XGBoost
    m = xgb.XGBRegressor(**xgb_params)
    m.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m.predict(X_val)
    test_xgb += m.predict(X_test) / 5
    
    # CatBoost
    m = CatBoostRegressor(**cat_params)
    m.fit(X_tr, y_tr)
    oof_cat[val_idx] = m.predict(X_val)
    test_cat += m.predict(X_test) / 5
    
    # RandomForest
    m = RandomForestRegressor(n_estimators=400, min_samples_leaf=1, n_jobs=-1, random_state=42)
    m.fit(X_tr, y_tr)
    oof_rf[val_idx] = m.predict(X_val)
    test_rf += m.predict(X_test) / 5
    
    r2_l = metrics.r2_score(y_val, oof_lgb[val_idx])
    r2_x = metrics.r2_score(y_val, oof_xgb[val_idx])
    r2_c = metrics.r2_score(y_val, oof_cat[val_idx])
    r2_r = metrics.r2_score(y_val, oof_rf[val_idx])
    print(f'Fold {fold+1}: LGB={r2_l:.5f} XGB={r2_x:.5f} CAT={r2_c:.5f} RF={r2_r:.5f}')

# Individual OOF scores
print(f'\nOOF R²:  LGB={metrics.r2_score(y, oof_lgb):.6f}  XGB={metrics.r2_score(y, oof_xgb):.6f}  CAT={metrics.r2_score(y, oof_cat):.6f}  RF={metrics.r2_score(y, oof_rf):.6f}')

Training K-Fold ensemble...
Fold 1: LGB=0.94503 XGB=0.94469 CAT=0.93750 RF=0.94229
Fold 2: LGB=0.94560 XGB=0.91307 CAT=0.93384 RF=0.94269
Fold 3: LGB=0.94428 XGB=0.94675 CAT=0.93745 RF=0.94227
Fold 4: LGB=0.93921 XGB=0.92941 CAT=0.93437 RF=0.93923
Fold 5: LGB=0.94415 XGB=0.94623 CAT=0.93905 RF=0.94197

OOF R²:  LGB=0.943717  XGB=0.935940  CAT=0.936450  RF=0.941727


In [14]:
# ── 12. Stacking with Ridge Meta-Learner ──────────────────────────────────────
# Stack OOF predictions
oof_stack = np.column_stack([oof_lgb, oof_xgb, oof_cat, oof_rf])
test_stack = np.column_stack([test_lgb, test_xgb, test_cat, test_rf])

# Ridge meta-learner
meta = Ridge(alpha=1.0)
meta.fit(oof_stack, y.values)
oof_meta = meta.predict(oof_stack)
meta_r2 = metrics.r2_score(y, oof_meta)
print(f'Ridge Stacking OOF R²: {meta_r2:.6f} | Score: {max(0,100*meta_r2):.2f}')
print(f'Meta coefficients: LGB={meta.coef_[0]:.4f} XGB={meta.coef_[1]:.4f} CAT={meta.coef_[2]:.4f} RF={meta.coef_[3]:.4f}')
print(f'Meta intercept: {meta.intercept_:.6f}')

# Test prediction via stacking
y_pred_stack = meta.predict(test_stack)

# Also try simple weighted average of best models
w = [metrics.r2_score(y, oof_lgb), metrics.r2_score(y, oof_xgb),
     metrics.r2_score(y, oof_cat), metrics.r2_score(y, oof_rf)]
w = np.array(w)
w = np.maximum(w, 0)  # clip negative weights
w = w / w.sum()
y_pred_weighted = (w[0]*test_lgb + w[1]*test_xgb + w[2]*test_cat + w[3]*test_rf)

# Also: top-2 ensemble (LGB + XGB or whichever is best)
scores_oof = [metrics.r2_score(y, oof_lgb), metrics.r2_score(y, oof_xgb),
              metrics.r2_score(y, oof_cat), metrics.r2_score(y, oof_rf)]
top2 = np.argsort(scores_oof)[-2:]
test_arrs = [test_lgb, test_xgb, test_cat, test_rf]
y_pred_top2 = (test_arrs[top2[0]] + test_arrs[top2[1]]) / 2

print(f'\nWeighted avg weights: LGB={w[0]:.3f} XGB={w[1]:.3f} CAT={w[2]:.3f} RF={w[3]:.3f}')

Ridge Stacking OOF R²: 0.944981 | Score: 94.50
Meta coefficients: LGB=0.5569 XGB=0.1998 CAT=-0.0512 RF=0.3096
Meta intercept: -0.001089

Weighted avg weights: LGB=0.251 XGB=0.249 CAT=0.249 RF=0.251


In [15]:
# ── 13. Save All Submissions ──────────────────────────────────────────────────
idx = test['Index'].values

# Submission 1: Ridge Stacking
sub1 = pd.DataFrame({'Index': idx, 'demand': y_pred_stack})
sub1.to_csv('submission_v3_stack.csv', index=False)

# Submission 2: Weighted Average
sub2 = pd.DataFrame({'Index': idx, 'demand': y_pred_weighted})
sub2.to_csv('submission_v3_weighted.csv', index=False)

# Submission 3: Top-2 Average
sub3 = pd.DataFrame({'Index': idx, 'demand': y_pred_top2})
sub3.to_csv('submission_v3_top2.csv', index=False)

print('Saved 3 submissions:')
print(f'  1. submission_v3_stack.csv    — Ridge stacking (OOF R²={meta_r2:.4f})')
print(f'  2. submission_v3_weighted.csv — weighted average')
print(f'  3. submission_v3_top2.csv     — top-2 model average')
print(f'\nAll shapes: {sub1.shape}')
display(sub1.head())

Saved 3 submissions:
  1. submission_v3_stack.csv    — Ridge stacking (OOF R²=0.9450)
  2. submission_v3_weighted.csv — weighted average
  3. submission_v3_top2.csv     — top-2 model average

All shapes: (41778, 2)


,Index,demand
0,0,0.053311
1,1,0.033632
2,2,0.038661
3,3,0.039469
4,4,0.041156


In [ ]:
# ── 14. Summary ───────────────────────────────────────────────────────────────
print('='*60)
print('v3 SUMMARY')
print('='*60)
print(f'LightGBM  OOF R²: {metrics.r2_score(y, oof_lgb):.6f} → Score: {max(0,100*metrics.r2_score(y, oof_lgb)):.2f}')
print(f'XGBoost   OOF R²: {metrics.r2_score(y, oof_xgb):.6f} → Score: {max(0,100*metrics.r2_score(y, oof_xgb)):.2f}')
print(f'CatBoost  OOF R²: {metrics.r2_score(y, oof_cat):.6f} → Score: {max(0,100*metrics.r2_score(y, oof_cat)):.2f}')
print(f'RF        OOF R²: {metrics.r2_score(y, oof_rf):.6f} → Score: {max(0,100*metrics.r2_score(y, oof_rf)):.2f}')
print(f'Ridge Stack  R²: {meta_r2:.6f} → Score: {max(0,100*meta_r2):.2f}')
print('='*60)
print('Submit all 3 files and keep the best score!')
print('  submission_v3_stack.csv    (usually best)')
print('  submission_v3_weighted.csv')
print('  submission_v3_top2.csv')